# 计算态演化:

In [6]:
import quante as qt
import numpy as np
# 拿到矩阵
L = 12
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis)
a = ham.to_matrix(basis, pauli=False, sparse=True)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

# 方法1:
res = qt.linalg.eigh(mat)
times = np.linspace(0,10,100)
qt.linalg.get_time_evolution_states_ED(state,*res,times)

array([[ 0.00335624+0.00606165j,  0.0048898 +0.00490894j,  0.00604849+0.00337989j, ...,  0.00508586-0.00470552j,  0.00360056-0.00591979j,  0.00183922-0.00668021j],
       [-0.01145467+0.0090065j , -0.00893959+0.01087129j, -0.00609528+0.01202976j, ...,  0.0022496 +0.01370704j,  0.0058218 +0.01275967j,  0.00901401+0.01092018j],
       [ 0.00826567+0.0048053j ,  0.01002091+0.00476051j,  0.01195179+0.00416268j, ..., -0.00079423+0.0112147j ,  0.00174312+0.01032498j,  0.00388811+0.00880588j],
       ...,
       [-0.01305153+0.00220341j, -0.01273198+0.00414389j, -0.01204725+0.00597486j, ...,  0.00022375+0.00682789j,  0.00212203+0.00639483j,  0.00377646+0.00548359j],
       [ 0.00607938+0.00661749j,  0.00764766+0.00570824j,  0.00906296+0.00443277j, ...,  0.00216894-0.0012144j ,  0.0021677 -0.00177j   ,  0.00199682-0.00239705j],
       [-0.01943596+0.01879979j, -0.01353566+0.02340886j, -0.00659764+0.02622327j, ...,  0.02289399+0.01438937j,  0.02596224+0.00755981j,  0.02704008+0.00015067j]])

## head

In [7]:
import quante as qt
import numpy as np

# 拿到矩阵
L = 12
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis, sparse=True)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

# 方法1:
import scipy.sparse as sp
res = sp.linalg.expm_multiply((-1j*mat), state, start=0, stop=10, num=100)
np.squeeze(res).T

array([[ 0.00335624+0.00606165j,  0.0048898 +0.00490894j,  0.00604849+0.00337989j, ...,  0.00508586-0.00470552j,  0.00360056-0.00591979j,  0.00183922-0.00668021j],
       [-0.01145467+0.0090065j , -0.00893959+0.01087129j, -0.00609528+0.01202976j, ...,  0.0022496 +0.01370704j,  0.0058218 +0.01275967j,  0.00901401+0.01092018j],
       [ 0.00826567+0.0048053j ,  0.01002091+0.00476051j,  0.01195179+0.00416268j, ..., -0.00079423+0.0112147j ,  0.00174312+0.01032498j,  0.00388811+0.00880588j],
       ...,
       [-0.01305153+0.00220341j, -0.01273198+0.00414389j, -0.01204725+0.00597486j, ...,  0.00022375+0.00682789j,  0.00212203+0.00639483j,  0.00377646+0.00548359j],
       [ 0.00607938+0.00661749j,  0.00764766+0.00570824j,  0.00906296+0.00443277j, ...,  0.00216894-0.0012144j ,  0.0021677 -0.00177j   ,  0.00199682-0.00239705j],
       [-0.01943596+0.01879979j, -0.01353566+0.02340886j, -0.00659764+0.02622327j, ...,  0.02289399+0.01438937j,  0.02596224+0.00755981j,  0.02704008+0.00015067j]])

## head

In [3]:
import quante as qt
import numpy as np
import time

# 拿到矩阵
L = 18
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis, sparse=True)
print("space dimension:", basis.Ns)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

# 方法1:
import scipy.sparse as sp

t = time.time()
res1 = sp.linalg.expm_multiply((-1j*mat), state, start=0, stop=10, num=100)
print(f"time scipy.expm_multiple: {time.time()-t:.2f}s")

t = time.time()
res2 = qt.linalg.expm_multiply((-1j*mat), state, start=0, stop=10, num=100, herm=True)
print(f"time quante.expm_multiple with cpu parallel: {time.time()-t:.2f}s")

t = time.time()
res3 = qt.linalg.expm_multiply((-1j*mat), state, start=0, stop=10, num=100, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

t = time.time()
res4 = qt.linalg.expm_multiply(mat, state, -1j, start=0, stop=10, num=100, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

print("norm difference between first two:", np.allclose(res1, res2))
print("norm difference between last two:", np.allclose(res2, res3))
print("norm difference between last two:", np.allclose(res3, res4))

space dimension: 262144
time scipy.expm_multiple: 23.79s
time quante.expm_multiple with cpu parallel: 17.55s


e:\hzhu\onedrive\share\python_library\quante\torch_utils\linalg\sparse.py:21: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  return tc.sparse_csr_tensor(tsr.indptr, tsr.indices, tsr.data, tsr.shape, dtype=dtype, device=device)


time quante.expm_multiple with gpu cuda: 3.14s
time quante.expm_multiple with gpu cuda: 1.30s
norm difference between first two: True
norm difference between last two: True
norm difference between last two: True


## head

In [4]:
import quante as qt
import numpy as np
import time

# 拿到矩阵
L = 22
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis, sparse=True)
print("space dimension:", basis.Ns)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

# 方法1:
import scipy.sparse as sp

t = time.time()
res1 = sp.linalg.expm_multiply((-1j*mat), state)
print(f"time scipy.expm_multiple: {time.time()-t:.2f}s")

t = time.time()
res2 = qt.linalg.expm_multiply((-1j*mat), state, herm=True)
print(f"time quante.expm_multiple with cpu parallel: {time.time()-t:.2f}s")

t = time.time()
res3 = qt.linalg.expm_multiply((-1j*mat), state, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

t = time.time()
res4 = qt.linalg.expm_multiply(mat, state, -1j, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

print("norm difference between first two:", np.allclose(res1, res2))
print("norm difference between last two:", np.allclose(res1, res3))
print("norm difference between last two:", np.allclose(res1, res4))

space dimension: 4194304
time scipy.expm_multiple: 16.09s
time quante.expm_multiple with cpu parallel: 8.84s
time quante.expm_multiple with gpu cuda: 0.95s
time quante.expm_multiple with gpu cuda: 0.47s
norm difference between first two: True
norm difference between last two: True
norm difference between last two: True


## head

In [1]:
import quante as qt
import numpy as np
import time

# 拿到矩阵
L = 23
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis, sparse=True)
print("space dimension:", basis.Ns)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

t = time.time()
res3 = qt.linalg.expm_multiply(mat, state, -1j, start=0, stop=10, num=100, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

space dimension: 8388608


e:\hzhu\onedrive\share\python_library\quante\torch_utils\linalg\sparse.py:21: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  return tc.sparse_csr_tensor(tsr.indptr, tsr.indices, tsr.data, tsr.shape, dtype=dtype, device=device)


time quante.expm_multiple with gpu cuda: 24.91s


## head

In [3]:
qt.basicfun.test_memory(res3)
import torch as tc
tc.cuda.empty_cache()

12.50 GB


## head

In [3]:
import quante as qt
import numpy as np
import time

# 拿到矩阵
L = 24
ham = qt.generate.operas.heisenberg_operator(L)
ham = ham.expandxy()
basis = qt.generate.basis.spin_basis(L)
# mat = ham.to_matrix(basis, sparse=True)
from quante.tensor.automata import get_sparse_matrix
mat = get_sparse_matrix(L, *ham.split_data(), pauli=False)
print("space dimension:", basis.Ns)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

t = time.time()
res3 = qt.linalg.expm_multiply(10*mat, state, scale=-1j, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

space dimension: 16777216
time quante.expm_multiple with gpu cuda: 28.32s


## head

In [5]:
import quante as qt
import numpy as np
import time

# 拿到矩阵
L = 24
ham = qt.generate.operas.heisenberg_operator(L)
ham = ham.expandxy()
basis = qt.generate.basis.spin_basis(L)
# mat = ham.to_matrix(basis, sparse=True)
from quante.tensor.automata import get_sparse_matrix
mat = get_sparse_matrix(L, *ham.split_data(), pauli=False)
print("space dimension:", basis.Ns)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

t = time.time()
res3 = qt.linalg.expm_multiply(0.1*mat, state, scale=-1j, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

space dimension: 16777216
time quante.expm_multiple with gpu cuda: 1.71s


## head

In [8]:
import quante as qt
import time

# 拿到矩阵
L = 16
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis, sparse=True)
print("space dimension:", basis.Ns)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42, dtype=np.complex128)

# 方法1:
import scipy.sparse as sp


t = time.time()
res2 = qt.linalg.expm_multiply((mat), state, start=1, stop=11, num=91)
print(f"time quante.expm_multiple with cpu parallel: {time.time()-t:.2f}s")
print(res2[-1,...].reshape(-1))

t = time.time()
res3 = qt.linalg.expm_multiply((mat), state, start=0, stop=10, num=95)
print(f"time quante.expm_multiple with cpu parallel: {time.time()-t:.2f}s")
print(res2[-1,...].reshape(-1))

space dimension: 65536


e:\hzhu\onedrive\share\python_library\quante\linalg\evolve.py:140: UserWarning: Using the q % s == 90 % 9 == 0 case, for best perfermance, change num slightly.
  return _expm_multiply_numba(lo, psi0, start=start, stop=stop, num=num, endpoint=endpoint, traceA=traceA)


time quante.expm_multiple with cpu parallel: 2.46s
[ 6.89243221e+14-4.00174737e+15j -9.51320889e+14-7.38116586e+13j -8.99574735e+14-8.61019848e+13j ...  6.98288778e+13-1.92433226e+14j  3.69413823e+13-2.40546913e+14j -4.01608056e+14+4.68143428e+13j]
time quante.expm_multiple with cpu parallel: 1.61s
[ 6.89243221e+14-4.00174737e+15j -9.51320889e+14-7.38116586e+13j -8.99574735e+14-8.61019848e+13j ...  6.98288778e+13-1.92433226e+14j  3.69413823e+13-2.40546913e+14j -4.01608056e+14+4.68143428e+13j]


## head

In [9]:
# 主播演化的方式
import quante as qt
from quante.torch_utils.linalg import evolve_engine, to_csr
import torch as tc
L = 10
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
tcmat = to_csr(ham.to_matrix(basis, sparse=True), device='cuda')
tcstate = tc.tensor(qt.generate.state.random(basis.Ns, seed=42), device='cuda')
eg = evolve_engine(tcmat, scale=-1j, herm=True)
for i in range(10):
    nextstate = eg(tcstate)
    assert tc.allclose(nextstate, tc.matrix_exp((-1j)*tcmat.to_dense()) @ tcstate)
    tcstate = nextstate

## head

In [1]:
# 拿到矩阵
import quante as qt
L = 24
ham = qt.generate.operas.heisenberg_operator(L)
ham = ham.expandxy()
basis = qt.generate.basis.spin_basis(L)
# mat = ham.to_matrix(basis, sparse=True)
from quante.tensor.automata import get_sparse_matrix
# qt.basicfun.test_time(get_sparse_matrix_cuda, L, *ham.split_data(), pauli=False)
qt.basicfun.test_time(get_sparse_matrix, L, *ham.split_data(), pauli=False)
# mat = get_sparse_matrix(L, *ham.split_data(), pauli=False)
# print("space dimension:", basis.Ns)


Timer unit: 1e-07 s

Total time: 3.42629 s
File: e:\hzhu\onedrive\share\python_library\quante\tensor\automata.py
Function: get_sparse_matrix at line 311

Line #      Hits         Time  Per Hit   % Time  Line Contents
   311                                           def get_sparse_matrix(
   312                                               L: int,
   313                                               hlocals: list[str],
   314                                               positions: list[tuple[int, ...]],
   315                                               coefficients: list[float],
   316                                               pauli: int = True,
   317                                           ) -> _sparse.csr_matrix:
   318                                               """
   319                                               利用 automata 生成稀疏矩阵
   320                                               
   321                                               Example:
   322             

## head

In [8]:
np.log2(5000)

12.287712379549449